In [0]:
from faker import Faker
from pyspark.sql import *
import random
from pyspark.sql.functions import *

In [0]:
fake = Faker("en_IN")

customer= []

for name in range(1, 21):
    customer.append(
        Row(
            customer_id = name,
            first_name = fake.first_name(),
            last_name = fake.last_name(),
            dob = fake.date_of_birth(),
            email = fake.email(),
            gender = random.choice(["Male", "Female"]),
            address = fake.address(),
            city = fake.city(),
            state = fake.state(),
            pincode = fake.postcode(),
            country = fake.country(),
            phone = "9" + "".join(random.choices("0123456789", k=9)),
            registration_date = str(fake.date_between(start_date='-2y', end_date='today'))
        )
    )

In [0]:
customer_incremental_df = spark.createDataFrame(customer)

In [0]:
customer_incremental_df.write \
    .mode("overwrite") \
        .format("delta") \
            .save("abfss://bronze@saretailsales.dfs.core.windows.net/customer_incremental/")

In [0]:
customer_incremental_df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- pincode: string (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- registration_date: string (nullable = true)



In [0]:
# Typecasting columns with correct data types
customer_incremental_df = customer_incremental_df.withColumn("phone", col("phone").cast("long"))

customer_incremental_df = customer_incremental_df.withColumn("registration_date", to_date(col("registration_date")))

customer_incremental_df = customer_incremental_df.withColumn("pincode", col("pincode").cast("long")) 

In [0]:
upsert_data = [
    (21, "Pooja", "Bal", "1992-05-06", "datewilliam@example.org", "Female",
     "H.No. 269 Sem Chowk Tezpur 034239", "Bangalore", "Karnataka", 359409,
     "India", "9476079365", "2025-09-26"),

    (22, "Omisha", "Borra", "1911-03-28", "miteshchad@example.net", "Female",
     "475, Mane Path, Bally 118666", "Patna", "West Bengal", 994997,
     "India", "9157004601", "2024-12-20"),

    (23, "Dayita", "Choudhury", "1985-01-11", "sabharwalekapad@example.org", "Male",
     "H.No. 86, Deshpande Chowk, Ratlam-135380", "Ranchi", "Bihar", 823598,
     "India", "9912835391", "2026-01-28"),

    (24, "Eshana", "Rastogi", "1943-01-23", "eshana@example.org", "Female",
     "353, Subramaniam Road, Pune 633943", "Medininagar", "Assam", 185270,
     "India", "9570340141", "2025-10-29"),

    (25, "Samaksh", "Sastry", "1944-12-16", "samaksh@example.org", "Male",
     "94, Chadha Ganj Panvel-330849", "Gulbarga", "Telangana", 819281,
     "India", "9494511064", "2026-05-02"),
    
    (26, "Omisha", "Borra", "1911-03-28", "miteshchad@example.net", "Female",
     "475, Mane Path, Bally 118666", "Patna", "West Bengal", 994997,
     "India", "9157004601", "2024-12-20"),

    (27, "Dayita", "Choudhury", "1985-01-11", "sabharwalekapad@example.org", "Male",
     "H.No. 86, Deshpande Chowk, Ratlam-135380", "Ranchi", "Bihar", 823598,
     "India", "9912835391", "2026-01-28"),

    (28, "Eshana", "Rastogi", "1943-01-23", "eshana@example.org", "Female",
     "353, Subramaniam Road, Pune 633943", "Medininagar", "Assam", 185270,
     "India", "9570340141", "2025-10-29"),
]

columns = [
    "customer_id", "first_name", "last_name", "dob", "email", "gender",
    "address", "city", "state", "pincode", "country", "phone", "registration_date"
]

upsert_df = spark.createDataFrame(upsert_data, columns)


In [0]:
upsert_df.printSchema()

upsert_df = upsert_df.withColumn("dob", to_date("dob"))
upsert_df = upsert_df.withColumn("pincode", col("pincode").cast("string"))

root
 |-- customer_id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- pincode: long (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- registration_date: string (nullable = true)



In [0]:
upsert_df.write \
    .mode("append") \
        .format("delta") \
            .save("abfss://bronze@saretailsales.dfs.core.windows.net/customer_incremental/")

In [0]:
new_incremental = spark.read \
    .format("delta") \
    .load("abfss://bronze@saretailsales.dfs.core.windows.net/customer_incremental/")

In [0]:
display(new_incremental)

customer_id,first_name,last_name,dob,email,gender,address,city,state,pincode,country,phone,registration_date
1,Sachi,Nagi,2011-10-18,osaran@example.org,Female,"H.No. 04 Wagle Circle, Silchar-671595",Mangalore,Chhattisgarh,059213,Serbia,9235013207,2024-09-23
2,Omisha,Brahmbhatt,1970-04-18,xnazareth@example.org,Female,22/861 Rege Bardhaman-411748,Gandhidham,Bihar,217268,Iran,9168476227,2026-01-02
3,Sara,Sarna,2014-06-10,vanyakamdar@example.com,Female,"H.No. 647, Hans Ganj, Gandhinagar 055587",Bhagalpur,Sikkim,616561,South Africa,9470666770,2026-04-25
4,Neha,Dani,1997-04-22,dharalka@example.org,Male,"297, Gill Path, Satna 334152",Katni,West Bengal,712637,Federated States of Micronesia,9838641782,2025-10-14
5,Wakeeta,Nair,1937-03-23,oyogi@example.com,Female,"08/62, Mall Street Mehsana-233339",Kolkata,Nagaland,837435,Spain,9695348102,2025-12-21
6,Daniel,Baral,1947-03-05,qrajagopal@example.org,Male,"375 Chakrabarti Chowk, Amroha-223665",Sambhal,Maharashtra,415602,Yemen,9213305730,2024-09-10
7,Jalsa,Memon,2010-06-22,gprasad@example.com,Male,68 Bhatia Circle Loni-153617,Purnia,Madhya Pradesh,108838,Iceland,9224759293,2025-01-24
8,Rachana,Hayre,1952-11-21,meghamagar@example.org,Male,"H.No. 89 Mittal Ganj, Shimla 910752",Allahabad,Haryana,803338,Nepal,9020479158,2024-07-22
9,Dalbir,Kala,1949-08-20,aggarwalnikita@example.com,Female,H.No. 902 Dewan Street Jammu-435109,Jalna,Manipur,943025,Monaco,9018224235,2024-08-07
10,Zehaan,Talwar,1911-12-12,dhaliwalrudra@example.org,Female,27/52 Mishra Chowk Navi Mumbai 482383,Warangal,Arunachal Pradesh,858992,Cameroon,9845656222,2025-04-26
